# Paper Event Selection

## Objective

Reconstruct the flare sample analyzed in Awasthi et al.

This notebook establishes the correspondence between the STIX flare
catalog and the flare events reported in the paper.

## Current understanding:

- Initial STIX detections in paper: 217 (215 from current API).
- Successfully characterized by paper: 203.
- Exact reason for exclusion of 14 events is **not explicitly stated** in the paper
  - (While its stated that 13 flares are ≥ C-class)

In [1]:
import pandas as pd

flare_df = pd.read_csv(
    "../data/processed/stix_flare_catalog_sep20_25_2021.csv"
)

flare_df["flare_id"] = flare_df["flare_id"].astype(str)

In [2]:
# To confirm that our catalog aligns with the paper's, we will search for the 1st entry in Table 1: 'SOL2021-09-21T09:46:22'

flare_df["peak_UTC"] = pd.to_datetime(flare_df["peak_UTC"])
flare_df["peak_UTC"]

0     2021-09-25 23:06:40.433
1     2021-09-25 22:04:20.427
2     2021-09-25 21:39:28.431
3     2021-09-25 21:13:08.428
4     2021-09-25 20:58:36.420
                ...          
210   2021-09-20 19:37:04.098
211   2021-09-20 19:30:00.098
212   2021-09-20 19:21:40.097
213   2021-09-20 19:04:04.095
214   2021-09-20 18:55:24.094
Name: peak_UTC, Length: 215, dtype: datetime64[us]

In [3]:
paper_time = pd.Timestamp("2021-09-21T09:46:22")

In [4]:
tolerance = pd.Timedelta(minutes = 5)

matches = flare_df[
    (flare_df["peak_UTC"] >= paper_time - tolerance) &
    (flare_df["peak_UTC"] <= paper_time + tolerance)
]

matches[[
    "flare_id",
    "start_UTC",
    "peak_UTC",
    "GOES_class",
    "GOES_flux"
]]

,flare_id,start_UTC,peak_UTC,GOES_class,GOES_flux
176,2109210950,2021-09-21T09:49:00.060,2021-09-21 09:50:20.060,B2.9,2.918430e-07
177,2109210943,2021-09-21T09:41:08.060,2021-09-21 09:43:00.060,B2.3,2.321424e-07


## Conclusion

Timestamp-based matching between the STIX flare catalog and the paper's named SOL
events proved unreliable. The catalog's `peak_UTC` field is generated by STIX's
automated detection algorithm and does not correspond to the manually-identified SXR
flux maximum used by Awasthi et al. (2021) to define their SOL event identifiers.

A 5-minute tolerance search around the first paper event (SOL2021-09-21T09:46:22)
returned two candidates (`2109210943` and `2109210950`), neither of which aligns
cleanly with the paper's timestamp.

**Decision:** This approach is parked. Reliable event cross-identification requires
light-curve shape comparison against the paper's Figure 3 panels, which in turn
requires the full characterization pipeline (background, onset, peak) to exist first.
This notebook will be revisited as an optional validation step after the pipeline
produces qf values for all 215 flares.

*Archived — not part of the active pipeline. See RESEARCH_LOG.md for full context.*